In [1]:
import json
import pandas as pd


def load_sentiment_results_df(jsonl_path: str) -> pd.DataFrame:
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            row = {"id": rec["id"], "message": rec["message"]}
            if rec["result"]:
                row.update(rec["result"])  # label, custom_label, reasoning
            else:
                row.update({"label": None, "custom_label": None, "reasoning": None})
            records.append(row)
    return pd.DataFrame(records).sort_values("id").reset_index(drop=True)


sentiment_jsonl_path = "../data/sub_classifications/sentiment_expression/raw.jsonl"
df_sentiment_results = load_sentiment_results_df(sentiment_jsonl_path)
df_sentiment_results.drop(columns=["id", "message"], inplace=True)
df_sentiment_results.rename(
    columns={"label": "sentiment_label", "reasoning": "sentiment_reasoning"},
    inplace=True,
)

In [2]:
df_classifications = pd.read_csv(
    "../data/classifications/classifications_for_analysis.csv"
)

df_sentiments = (
    df_classifications[df_classifications["sub_category"] == "7.5 Sentiment Expression"]
    .copy()
    .reset_index(drop=True)
)

df_sentiments = pd.concat([df_sentiments, df_sentiment_results], axis=1)

In [3]:
import ast
from collections import Counter

# Parse labels column
df_sentiments["labels_parsed"] = df_sentiments["labels"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_sentiments["label_count"] = df_sentiments["labels_parsed"].apply(len)
df_sentiments["label_type"] = df_sentiments["label_count"].apply(
    lambda x: "single_label" if x == 1 else "multi_label"
)

for sentiment in ["positive", "negative"]:
    subset = df_sentiments[df_sentiments["sentiment_label"] == sentiment]
    total = len(subset)
    print(f"{'='*50}")
    print(f"Sentiment: {sentiment} (total: {total})")
    print(f"{'='*50}")

    # Single vs Multi label
    print("\n--- Single vs Multi Label ---")
    for label_type, count in subset["label_type"].value_counts().items():
        print(f"  {label_type}: {count} ({count/total*100:.1f}%)")

    # Other sub_category distribution (excluding Sentiment Expression)
    other_labels = []
    for _, row in subset.iterrows():
        for label in row["labels_parsed"]:
            sub = label.get("sub_category", "")
            if sub != "7.5 Sentiment Expression":
                other_labels.append(sub)

    print("\n--- Other Sub-category Distribution (excl. Sentiment Expression) ---")
    if other_labels:
        counts = Counter(other_labels)
        total_other = sum(counts.values())
        for k, v in counts.most_common():
            print(f"  {k}: {v} ({v/total_other*100:.1f}%)")
        print(f"  Total other labels: {total_other}")
    else:
        print("  No other labels found")

    # Top 10 custom_label
    print("\n--- Top 10 Custom Labels ---")
    top10 = subset["custom_label"].value_counts().head(10)
    for label, count in top10.items():
        print(f"  {label}: {count} ({count/total*100:.1f}%)")

Sentiment: positive (total: 303)

--- Single vs Multi Label ---
  multi_label: 202 (66.7%)
  single_label: 101 (33.3%)

--- Other Sub-category Distribution (excl. Sentiment Expression) ---
  7.1 Confirmation: 81 (36.5%)
  7.2 Continuation: 28 (12.6%)
  1.2 Iterative Modification: 19 (8.6%)
  6.2 Toolchain Operation: 18 (8.1%)
  6.1 Documentation: 15 (6.8%)
  4.1 Information Injection: 13 (5.9%)
  3.2 Project Comprehension: 10 (4.5%)
  4.2 Behavior Specification: 9 (4.1%)
  2.2 Symptom Description: 7 (3.2%)
  1.1 New Implementation: 6 (2.7%)
  3.1 Planning & Decision Consultation: 6 (2.7%)
  2.1 Log Paste: 3 (1.4%)
  1.3 Alignment Correction: 2 (0.9%)
  5.2 Runtime Inspection: 2 (0.9%)
  3.3 General Knowledge Query: 2 (0.9%)
  5.1 Code Review: 1 (0.5%)
  Total other labels: 222

--- Top 10 Custom Labels ---
  gratitude: 81 (26.7%)
  satisfaction: 43 (14.2%)
  excitement: 34 (11.2%)
  praise: 29 (9.6%)
  greeting: 19 (6.3%)
  approval: 16 (5.3%)
  enthusiasm: 15 (5.0%)
  appreciation: 14